# Урок 12 - Намаляване на историята на чата с агентски блокнот

Този бележник демонстрира как да управляваме контекст в дълги разговори с помощта на Microsoft Agent Framework. С разрастването на разговорите броят на токените се увеличава — в крайна сметка надвишавайки контекстното прозорче на модела. Ние решаваме това с помощта на **шаблон за обобщаване на контекста** и **агентски блокнот** за постоянна памет.

## Какво ще научите:
1. **Защо управлението на контекста е важно**: Разбиране на лимитите за токени и контекстните прозорци
2. **Агенти, осведомени за контекста**: Създаване на агенти, които управляват собствения си контекст на разговор
3. **Шаблон за обобщаване на контекста**: Използване на инструменти за съкращаване на историята на разговора
4. **Агентски блокнот**: Постоянна памет, която оцелява при намаляване на контекста

## Предварителни изисквания:
- Настройка на Azure OpenAI с конфигурирани променливи на средата
- Разбиране на основните концепции за агенти от предишни уроци


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Защо управлението на контекста има значение

Всеки LLM има крайно **контекстно поле** — максималният брой токени, които може да обработи в една заявка. С напредването на многократен разговор:

- **Броят токени расте линейно** с всяко съобщение от потребителя и отговора на асистента.
- **Токените на подканата доминират разходите**, защото цялата история се изпраща наново всеки ход.
- Накрая разговорът **превишава контекстния прозорец** и моделът или прекъсва, или дава грешка.

### Стратегии за управление на контекста

| Стратегия | Как работи | Компромис |
|---|---|---|
| **Съкратяване** | Премахват се най-старите съобщения | Загуба на ранния контекст |
| **Обобщаване** | Съкращаване на по-стари съобщения в обобщение | Някои детайли се губят, но ключовите точки се запазват |
| **Бележник / Външна памет** | Съхраняване на ключови факти извън разговора | Изисква повиквания на инструменти, но запазва информация при всяко съкращаване |

В този бележник комбинираме **обобщаване** с **инструмент за бележник**, така че агентът да може да поддържа непрекъснатост дори когато историята на разговора е съкратена.


## Създаване на агент, осъзнат за контекста


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Симулиране на дълъг разговор

Нека преминем през многократно провеждан разговор, за да видим как контекстът се натрупва. Агентът трябва да запази ключови детайли (предпочитания, бюджет, дати на пътуване) през времето на разговора и да демонстрира последователност.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Обърнете внимание как агентът запазва контекста от предишните ходове — той знае за Япония, суши, храмове, фотография, бюджета от $3000, пътуването на поединец и периода през април. В кратък разговор това работи добре, но с разрастването на разговора цялата история става скъпа за повторно изпращане.

Нека продължим разговора с още ходове, за да видим натрупването на контекст:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Шаблон за обобщаване на контекста

С разрастването на разговора можем да използваме **инструмент за обобщаване**, който кондензира натрупания контекст в компактен формат. Агента използва този инструмент, за да записва ключови предпочитания, така че дори и по-старите съобщения да бъдат премахнати, съществената информация се запазва.

Този шаблон е основата за по-сложно съкращаване на историята:
1. Агента идентифицира ключови факти от разговора
2. Извиква инструмента за обобщаване, за да ги запази
3. По-старите съобщения могат да бъдат безопасно премахнати, защото обобщението запазва същественото

По-долу дефинираме инструмент `summarize_preferences`, който агентът може да използва, за да запише компактен резюме на наученото.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Обобщение

В този урок научихте как да управлявате контекста в дълги разговори с агент, използвайки Microsoft Agent Framework:

### Основни понятия
- **Контекстните прозорци са ограничени** — всеки токен в историята на разговора струва пари и се брои към лимита.
- **Инструменти за обобщаване** позволяват на агента да свива натрупания контекст в компактни резюмета, намалявайки използването на токени, като същевременно запазва съществената информация.
- **Бележници на агента** предоставят постоянна външна памет, която оцелявa при всяко съкращаване на разговора.

### Какво създадохте
- **Агент, осъзнаващ контекста**, който поддържа непрекъснатост в многократни разговори
- **Инструмент за обобщаване** (`summarize_preferences`), който записва ключови потребителски детайли в компактен формат
- **Многократен разговор**, демонстриращ запазване на контекста и обработка на промени

### Приложения в реалния свят
- **Ботове за обслужване на клиенти**: Запомняне на предпочитанията през дълги сесии за поддръжка
- **Лични асистенти**: Следене на текущи проекти без повторно обясняване на контекста
- **Образователни наставници**: Поддържане на напредъка на ученика през много взаимодействия

### Следващи стъпки
- Реализирайте пълен инструмент за бележник с файлово базирана персистентност
- Добавете автоматично съкращаване на историята след обобщаването
- Комбинирайте с векторни бази данни за семантично търсене в паметта
- Създайте агенти, които могат да подновяват разговори дни по-късно с пълен контекст


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
